# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Mapa (se eligieron posiciones razonables para el ejercicio)
        self.start = (4, 0)
        self.walls = { (1,1), (2,1), (3,1), (1,3), (2,3), (3,3) }
        self.slippery_states = { (2,2), (3,2) }

        self.terminal_states = {
            (0,5): 10.0,   # entrega +10 (top-right)
            (4,5): 2.0,    # estación de carga +2 (bottom-right)
            (0,0): -10.0,  # peligro mortal -10 (top-left)
        }

        self.danger_states = {
            (2,5): -3.0,
            (3,0): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if r < 0 or r >= self.height or c < 0 or c >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        S = []
        for r in range(self.height):
            for c in range(self.width):
                s = (r,c)
                if self.is_valid_state(s):
                    S.append(s)
        return S

    def is_terminal(self, state):
        return state in self.terminal_states or state in self.danger_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]
        """
        # Estados terminales son absorbentes
        if self.is_terminal(state):
            return [(state, 1.0)]

        # Probabilidades según tipo de piso
        if state in self.slippery_states:
            p_forward = 0.6
            p_left = 0.2
            p_right = 0.2
        else:
            p_forward = 0.9
            p_left = 0.05
            p_right = 0.05

        def move(s, a):
            nr = s[0] + a[0]
            nc = s[1] + a[1]
            ns = (nr, nc)
            if not self.is_valid_state(ns):
                return s
            return ns

        forward = move(state, action)
        left_a = (-action[1], action[0])
        right_a = (action[1], -action[0])
        left = move(state, left_a)
        right = move(state, right_a)

        probs = {}
        for ns, p in ((forward, p_forward), (left, p_left), (right, p_right)):
            probs[ns] = probs.get(ns, 0.0) + p

        return list(probs.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    """Devuelve sum_{s'} T(s,a,s') * V(s')"""
    transitions = grid.get_transition_probs(state, action)
    return sum(p * V.get(s2, 0.0) for s2, p in transitions)


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}

    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            best = None
            for a in grid.actions:
                ev = expected_next_value(grid, s, a, V)
                if best is None or ev > best:
                    best = ev
            V_new[s] = grid.get_reward(s) + grid.gamma * (best if best is not None else 0.0)
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V, it
    return V, max_iter


def extract_policy(grid, V):
    pi = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        best_a = None
        best_val = None
        for a in grid.actions:
            ev = expected_next_value(grid, s, a, V)
            if best_val is None or ev > best_val:
                best_val = ev
                best_a = a
        pi[s] = best_a
    return pi



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    S = grid.states()
    V = {s: 0.0 for s in S}
    for s in S:
        if grid.is_terminal(s):
            V[s] = grid.get_reward(s)
    for it in range(1, max_iter + 1):
        delta = 0.0
        V_new = V.copy()
        for s in S:
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
                continue
            a = policy.get(s, grid.actions[0])
            transitions = grid.get_transition_probs(s, a)
            expected = sum(p * V[t] for t, p in transitions)
            V_new[s] = grid.get_reward(s) + grid.gamma * expected
            delta = max(delta, abs(V_new[s] - V[s]))
        V = V_new
        if delta < threshold:
            return V
    return V


def policy_improvement(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        best_a = None
        best_val = None
        for a in grid.actions:
            ev = expected_next_value(grid, s, a, V)
            if best_val is None or ev > best_val:
                best_val = ev
                best_a = a
        policy[s] = best_a
    return policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. política inicial arbitraria
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue
        policy[s] = grid.actions[0]

    history = []
    for it in range(1, max_iter + 1):
        V = policy_evaluation(grid, policy, threshold=threshold)
        new_policy = policy_improvement(grid, V)
        # count changes
        changes = sum(1 for s in new_policy if (s not in policy) or (policy[s] != new_policy[s]))
        history.append(changes)
        if changes == 0:
            return new_policy, V, history
        policy = new_policy
    return policy, V, history



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:


## Respuestas de la Parte 5

1. Desde `START`, la politica optima intenta llegar a la entrega `+10`. Aunque la carga `+2` es mas cercana, el premio de la entrega compensa el costo de los pasos y el riesgo del recorrido.
2. Una recompensa menor puede ser optima si esta mas cerca o tiene menor riesgo. El retorno tambien depende del costo por paso, del descuento `gamma` y de las probabilidades de transicion.
3. El piso resbaloso cambia las decisiones en los estados marcados como resbalosos, especialmente cuando una desviacion puede llevar a una pared, a un peligro o a una ruta de menor valor.
4. El costo `-1` penaliza cada movimiento, evita que el robot deambule y favorece rutas cortas. Por eso una recompensa grande pero distante no siempre domina a una recompensa menor y cercana.
5. Las probabilidades dependen del estado actual: en piso normal son `0.90`, `0.05` y `0.05`; en piso resbaloso son `0.60`, `0.20` y `0.20`. Las colisiones acumulan probabilidad en el estado actual.

### Experimentos

- **A (`living_reward = -0.1`):** caminar cuesta menos, por lo que el robot puede aceptar una ruta mas larga; desde `START` sigue dirigiendose a la entrega `+10`.
- **B (piso resbaloso con probabilidad deseada `0.40`):** aumenta la incertidumbre y baja el valor de los estados resbalosos; la politica debe evitar desviaciones hacia peligros y paredes. En este mapa, desde `START`, sigue prefiriendo la entrega.
- **C (`gamma = 0.99`):** se descuentan menos las recompensas futuras, asi que la entrega distante `+10` resulta mas atractiva.
- **Bonus:** el cambio entre carga y entrega aparece aproximadamente cerca de `living_reward = -1.035` con `gamma = 0.9`. Por debajo de ese valor conviene la carga; por encima, la entrega.